# CODE 1.3: Split Data

> **Goal:** split the photos into three parts: **train** (to learn), **validation** (to choose settings) and **test** (for the final score).
>
> - **Input:** `data/raw/chewinggum/image_anno.csv`, VisA's split files in `data/raw/split_csv/`, and `splits/same_gum_photos.csv` (from notebook 1.2)
> - **Output:** `splits/split_1.csv`, `splits/split_2.csv` and `splits/split_3.csv`
>
> The photos are not moved or copied. A split file only says which part each photo belongs to. Photos that show the same gum as another photo (notebook 1.2) are left out.

## Import Libraries & Variables

### Import Libraries

> - `os`: works with folders and file paths.
> - `pandas`: reads and saves tables.
> - `train_test_split` (from scikit-learn): shuffles rows and splits them into two parts.

In [1]:
import os

import pandas as pd
from sklearn.model_selection import train_test_split

### Variables

#### Folder Paths

> This code finds the main project folder. It starts in the folder of this notebook and goes up until it finds the `data` folder, so the paths work wherever the notebook is.
>
> - `ROOT_DIR`: the main project folder.
> - `RAW_DIR`: the original data. We only read from it.
> - `GUM_DIR`: the chewing gum photos, masks and photo list.
> - `SPLIT_CSV_DIR`: VisA's own split files.
> - `SPLITS_DIR`: where our split files are saved.
> - `SAME_GUM_FILE`: the list of photos that show the same gum (notebook 1.2).
>
> If a file is missing, the code stops with a message that says what to do.

In [2]:
ROOT_DIR = os.getcwd()
while not os.path.isdir(os.path.join(ROOT_DIR, "data")) and ROOT_DIR != os.path.dirname(ROOT_DIR):
    ROOT_DIR = os.path.dirname(ROOT_DIR)

RAW_DIR = os.path.join(ROOT_DIR, "data", "raw")
GUM_DIR = os.path.join(RAW_DIR, "chewinggum")
SPLIT_CSV_DIR = os.path.join(RAW_DIR, "split_csv")
SPLITS_DIR = os.path.join(ROOT_DIR, "splits")
SAME_GUM_FILE = os.path.join(SPLITS_DIR, "same_gum_photos.csv")

if not os.path.exists(os.path.join(GUM_DIR, "image_anno.csv")):
    raise FileNotFoundError("Data not found. Copy the 'chewinggum' folder into data/raw/ (see README).")
if not os.path.exists(os.path.join(SPLIT_CSV_DIR, "1cls.csv")):
    raise FileNotFoundError("VisA split files not found in data/raw/split_csv/ (see README).")
if not os.path.exists(SAME_GUM_FILE):
    raise FileNotFoundError("splits/same_gum_photos.csv not found. Run notebook 1.2 first.")
print("Project folder found:", os.path.basename(ROOT_DIR))

Project folder found: visual-defect-inspection


#### Split Settings

> - How many normal and defect photos go to test and validation. These sizes are our choice. All other photos (except the left-out ones) go to train.
> - `LABEL_STEPS`: how many labeled defect photos YOLO gets in each experiment.
> - `SEEDS`: one split is made for each seed. A seed is the starting number of the random shuffle. The same seed always gives the same shuffle, so anyone can make the exact same split again.

In [3]:
N_TEST_NORMAL = 100
N_TEST_DEFECT = 50
N_VAL_NORMAL = 50
N_VAL_DEFECT = 10

LABEL_STEPS = [5, 10, 20, 40]
SEEDS = [1, 2, 3]

## Functions

### Count Photos in a VisA Split

> `count_photos()` counts the photos of one part (`train` or `test`) with one label (`normal` or `anomaly`) in one VisA split table.

In [4]:
def count_photos(gum, part, label):
    return int(((gum["split"] == part) & (gum["label"] == label)).sum())

### Photos in a Part of a VisA Split

> `photos_in()` reads one VisA split file and gives the set of chewing gum photos in one part.

In [5]:
def photos_in(name, part):
    visa = pd.read_csv(os.path.join(SPLIT_CSV_DIR, f"{name}.csv"))
    gum = visa[visa["object"] == "chewinggum"]
    return set(gum.loc[gum["split"] == part, "image"])

### Type Group of a Photo

> `type_group_of()` gives the group of a photo for the even split (see "Give Each Photo a Type Group"): `normal` for a normal photo, otherwise its rarest defect type. `type_counts` says how many photos have each type.

In [6]:
def type_group_of(label, type_counts):
    if label == "normal":
        return "normal"
    return min(label.split(","), key=lambda t: type_counts[t])

### Make One Split

> `make_split()` makes one split in five steps:
>
> 1. **Left-out photos:** photos with `left_out` = `True` get the part `excluded` and are not used below.
> 2. **Normal photos:** shuffle, then take `n_test_normal` photos for test and `n_val_normal` for validation. The rest go to train.
> 3. **Defect photos:** the same with `n_test_defect` and `n_val_defect`, but every type group is split in the same proportions.
> 4. **Order the train defect photos** with numbers 1, 2, 3, and so on, taking turns between type groups. So photos 1 to 5 come from 5 different groups.
> 5. **Mark the YOLO photos:** `yolo_5` is `True` for the photos YOLO may use when it gets 5 defect photos: all normal train photos + defect photos 1 to 5. The same for `yolo_10`, `yolo_20` and `yolo_40`.
>
> Because of step 4, every bigger set includes the smaller one: the 10 photos are the first 5 plus 5 more. So if YOLO gets better with 10 photos, it is because of the extra photos, not because of a different random pick.

In [7]:
def make_split(photos, seed, n_test_normal, n_test_defect, n_val_normal, n_val_defect, label_steps):
    # 1. Left-out photos are not used for train, validation or test
    usable = photos[~photos["left_out"]]
    normal = usable[~usable["is_defect"]]
    defect = usable[usable["is_defect"]]

    # 2. Normal photos: a plain random split
    normal_rest, normal_test = train_test_split(normal, test_size=n_test_normal, random_state=seed)
    normal_train, normal_val = train_test_split(normal_rest, test_size=n_val_normal, random_state=seed)

    # 3. Defect photos: every type group is split in the same proportions
    defect_rest, defect_test = train_test_split(defect, test_size=n_test_defect,
                                                stratify=defect["type_group"], random_state=seed)
    defect_train, defect_val = train_test_split(defect_rest, test_size=n_val_defect,
                                                stratify=defect_rest["type_group"], random_state=seed)

    # 4. Order the train defect photos: take turns between type groups
    queues = [group.sample(frac=1, random_state=seed).index.tolist()
              for _, group in defect_train.groupby("type_group")]
    order = []
    while any(queues):
        for queue in queues:
            if queue:
                order.append(queue.pop(0))
    order_number = {row: i + 1 for i, row in enumerate(order)}

    # Write the part name for every photo. The left-out photos keep the part "excluded"
    result = photos.copy()
    result["split"] = "excluded"
    for part, normal_part, defect_part in [("train", normal_train, defect_train),
                                           ("val", normal_val, defect_val),
                                           ("test", normal_test, defect_test)]:
        result.loc[normal_part.index, "split"] = part
        result.loc[defect_part.index, "split"] = part
    result["label_order"] = pd.Series(order_number, dtype="Int64").reindex(result.index)

    # 5. Mark the photos YOLO may use in each experiment
    order_or_never = result["label_order"].fillna(10**6)   # photos without a number are never picked
    for k in label_steps:
        result[f"yolo_{k}"] = (result["split"] == "train") & (~result["is_defect"] | (order_or_never <= k))

    columns = ["image", "mask", "is_defect", "defect_types", "type_group", "split", "same_gum_as", "label_order"]
    return result[columns + [f"yolo_{k}" for k in label_steps]]

## Load Data

### Load the Photo List

> We read the photo list and add two columns:
>
> - `is_defect`: `True` for a defect photo
> - `defect_types`: the defect type(s), empty for normal photos

In [8]:
photos = pd.read_csv(os.path.join(GUM_DIR, "image_anno.csv")).fillna("")
photos["is_defect"] = photos["label"] != "normal"
photos["defect_types"] = photos["label"].where(photos["is_defect"], "")

print(photos["is_defect"].map({False: "normal", True: "defect"}).rename("kind").value_counts().to_string())
print("total    ", len(photos))

kind
normal    503
defect    100
total     603


### Load the Photos of the Same Gum

> Notebook 1.2 found photos that show the same gum twice. For each such pair we keep one photo and leave the other out. We add two columns:
>
> - `left_out`: `True` for a photo we leave out
> - `same_gum_as`: for a left-out photo, the photo we keep of the same gum (empty for all other photos)

In [9]:
same_gum = pd.read_csv(SAME_GUM_FILE).fillna({"same_gum_as": ""})
left_out = same_gum[~same_gum["keep"]]

photos["left_out"] = photos["image"].isin(left_out["image"])
photos["same_gum_as"] = photos["image"].map(dict(zip(left_out["image"], left_out["same_gum_as"]))).fillna("")

print(f"Photos left out: {photos['left_out'].sum()} "
      f"({(photos['left_out'] & ~photos['is_defect']).sum()} normal, {(photos['left_out'] & photos['is_defect']).sum()} defect)")
photos.loc[photos["left_out"], ["image", "same_gum_as"]]

Photos left out: 6 (6 normal, 0 defect)


,image,same_gum_as
113,chewinggum/Data/Images/Normal/113.JPG,chewinggum/Data/Images/Normal/112.JPG
216,chewinggum/Data/Images/Normal/216.JPG,chewinggum/Data/Images/Normal/215.JPG
265,chewinggum/Data/Images/Normal/265.JPG,chewinggum/Data/Images/Normal/264.JPG
300,chewinggum/Data/Images/Normal/300.JPG,chewinggum/Data/Images/Normal/299.JPG
351,chewinggum/Data/Images/Normal/351.JPG,chewinggum/Data/Images/Normal/350.JPG
501,chewinggum/Data/Images/Normal/501.JPG,chewinggum/Data/Images/Normal/500.JPG


## VisA's Own Splits

### What Are 1-Class and 2-Class?

> Before training a model we split the photos: **train** photos to learn from, and **test** photos to measure how good the model is. VisA gives three ready-made splits, one file each in `data/raw/split_csv/`. Their names say how the model is trained:
>
> - **1-class** (`1cls.csv`): the model learns from **one class** only, the normal photos. All defect photos are kept for the test. This fits methods like PatchCore, which learn what "normal" looks like and flag anything different.
> - **2-class** (`2cls_highshot.csv` and `2cls_fewshot.csv`): the model learns from **two classes**, normal and defect photos. This fits methods like YOLO, which need defect examples to learn from.
> - **high-shot** and **few-shot**: how many defect photos the model gets to learn from. A "shot" is one example, so high-shot means many defect examples and few-shot means only a few.
>
> The next two steps count what each file does with the chewing gum photos.

### Count VisA's Splits for Chewing Gum

> For each file we count the chewing gum photos in train and test. `parts in the file` shows which parts the file has. The last column shows which share of the defect photos goes to train.

In [10]:
rows = []
for name in ["1cls", "2cls_highshot", "2cls_fewshot"]:
    visa = pd.read_csv(os.path.join(SPLIT_CSV_DIR, f"{name}.csv"))
    gum = visa[visa["object"] == "chewinggum"]
    rows.append({
        "VisA split": name,
        "parts in the file": ", ".join(sorted(gum["split"].unique())),
        "train normal": count_photos(gum, part="train", label="normal"),
        "train defect": count_photos(gum, part="train", label="anomaly"),
        "test normal": count_photos(gum, part="test", label="normal"),
        "test defect": count_photos(gum, part="test", label="anomaly"),
    })

visa_splits = pd.DataFrame(rows).set_index("VisA split")
share = visa_splits["train defect"] / (visa_splits["train defect"] + visa_splits["test defect"])
visa_splits["defect photos in train"] = share.map("{:.0%}".format)
visa_splits

,parts in the file,train normal,train defect,test normal,test defect,defect photos in train
VisA split,,,,,,
1cls,"test, train",453,0,50,100,0%
2cls_highshot,"test, train",302,60,201,40,60%
2cls_fewshot,"test, train",101,20,402,80,20%


### Can Both Models Share a Test Set?

> For a fair comparison, PatchCore and YOLO must be scored on the same test photos, and neither model may have learned from them.
>
> Suppose PatchCore uses the 1-class file and YOLO uses a 2-class file. For each 2-class file the table counts:
>
> - its test photos, and how many of them are **train** photos in `1cls` (so PatchCore would have learned from them),
> - how many `1cls` test photos are **train** photos in that 2-class file (so YOLO would have learned from them).

In [11]:
rows = []
for name in ["2cls_highshot", "2cls_fewshot"]:
    test_photos = photos_in(name, part="test")
    rows.append({
        "2-class file": name,
        "its test photos": len(test_photos),
        "of these, train photos in 1cls": len(test_photos & photos_in("1cls", part="train")),
        "1cls test photos that are its train photos": len(photos_in("1cls", part="test") & photos_in(name, part="train")),
    })
pd.DataFrame(rows).set_index("2-class file")

,its test photos,"of these, train photos in 1cls",1cls test photos that are its train photos
2-class file,,,
2cls_highshot,241,180,89
2cls_fewshot,482,364,32


> **What we see:**
>
> - `1cls`: train has 0 defect photos, so YOLO would have nothing to learn from.
> - `2cls_highshot` puts 60% of the defect photos in train (60 photos) and `2cls_fewshot` 20% (20 photos). That is only two fixed amounts. We want to try 5, 10, 20 and 40 to see how many YOLO really needs.
> - There is no fair shared test set: 180 of the 241 high-shot test photos are train photos in `1cls`, so a PatchCore trained with `1cls` would be tested on photos it learned from. The other way round, 89 of the `1cls` test photos are train photos in `2cls_highshot`.
> - Every file only has `test` and `train`, no validation part to choose the PASS/REJECT limit. Choosing the limit on the test photos would make the score look better than it really is.
> - VisA's splits also do not know about the photos of the same gum (notebook 1.2).
>
> So we make our own split: one test set for both models, a validation part, 5, 10, 20 or 40 defect photos for YOLO, and no gum in two parts. Because of that, our scores cannot be compared one-to-one with scores in VisA papers.

## Our Split

### The Plan

| Part | Normal photos | Defect photos | Used for |
| --- | --- | --- | --- |
| `test` | `N_TEST_NORMAL` = 100 | `N_TEST_DEFECT` = 50 | the final score of both models, on the same photos |
| `val` | `N_VAL_NORMAL` = 50 | `N_VAL_DEFECT` = 10 | choosing the score limit between PASS and REJECT |
| `train` | the rest | the rest | PatchCore learns from the normal photos only. YOLO learns from the normal photos + 5, 10, 20 or 40 defect photos. |
| `excluded` | the left-out photos | the left-out photos | nothing: they show the same gum as a photo we keep |

> Test and validation sizes are our choice (see Split Settings). Train gets the rest. With the counts printed in "Load Data", that is 503 − 6 − 100 − 50 = 347 normal photos and 100 − 50 − 10 = 40 defect photos. The preview in "Make 3 Splits" below shows the real counts.
>
> We make this split **3 times**, once for each seed in `SEEDS`. If a result shows up in all three, it is not luck.

### Give Each Photo a Type Group

> We want every defect type in train, validation and test. With a plain shuffle, a rare type could end up almost only in one part.
>
> So each defect photo gets a **type group**: its rarest defect type. Then every type group is split in the same proportions. This is called a *stratified* split. Normal photos are in the type group `normal`.

In [12]:
type_counts = photos.loc[photos["is_defect"], "label"].str.split(",").explode().value_counts()
print("Photos per defect type:")
print(type_counts.to_string())

photos["type_group"] = photos["label"].map(lambda label: type_group_of(label, type_counts=type_counts))
print("\nPhotos per type group:")
print(photos["type_group"].value_counts().to_string())
print("\nDefect photos in all defect type groups together:", (photos["type_group"] != "normal").sum())

Photos per defect type:
label
chunk of gum missing    47
corner missing          36
small cracks            29
similar colour spot     25
scratches               16

Photos per type group:
type_group
normal                  503
small cracks             25
similar colour spot      23
corner missing           20
scratches                16
chunk of gum missing     16

Defect photos in all defect type groups together: 100


> **What we see:** every defect photo is in exactly one type group, so the groups add up to 100. *scratches* is the rarest type (16 photos), so all 16 photos with scratches are in the `scratches` group. *chunk of gum missing* is the most common type (47 photos), but only 16 photos are in its group: most of those photos also have a rarer type, and the rarer type decides the group.

### Make 3 Splits (Preview)

> We make the 3 splits in memory first and look at the counts before anything is saved. Every split should have the same counts: the plan above. Only *which* photos go where changes with the seed.

In [13]:
splits = {seed: make_split(photos, seed=seed, n_test_normal=N_TEST_NORMAL, n_test_defect=N_TEST_DEFECT,
                           n_val_normal=N_VAL_NORMAL, n_val_defect=N_VAL_DEFECT, label_steps=LABEL_STEPS)
          for seed in SEEDS}

for seed, split in splits.items():
    kind = split["is_defect"].map({False: "normal", True: "defect"}).rename("kind")
    table = pd.crosstab(split["split"], kind).reindex(index=["train", "val", "test", "excluded"],
                                                      columns=["normal", "defect"], fill_value=0)
    print(f"split_{seed}")
    print(table.to_string(), "\n")

split_1
kind      normal  defect
split                   
train        347      40
val           50      10
test         100      50
excluded       6       0 

split_2
kind      normal  defect
split                   
train        347      40
val           50      10
test         100      50
excluded       6       0 

split_3
kind      normal  defect
split                   
train        347      40
val           50      10
test         100      50
excluded       6       0 



### Save the Splits

> Each split is saved as a CSV file in `splits/`. Saving again with the same seeds makes exactly the same files.

In [14]:
os.makedirs(SPLITS_DIR, exist_ok=True)
for seed, split in splits.items():
    path = os.path.join(SPLITS_DIR, f"split_{seed}.csv")
    split.to_csv(path, index=False)
    print("Saved:", os.path.relpath(path, ROOT_DIR), f"({len(split)} rows)")

Saved: splits\split_1.csv (603 rows)
Saved: splits\split_2.csv (603 rows)
Saved: splits\split_3.csv (603 rows)


## Result

### Look at One Split File

> Each split file has one row per photo and these columns:
>
> - `image`, `mask`: where the files are, inside `data/raw`
> - `is_defect`: `True` for a defect photo
> - `defect_types`: the defect type(s)
> - `type_group`: the type group used for the even split (see "Give Each Photo a Type Group")
> - `split`: `train`, `val`, `test` or `excluded`
> - `same_gum_as`: for an excluded photo, the photo we keep of the same gum
> - `label_order`: the order number of the train defect photos
> - `yolo_5`, `yolo_10`, `yolo_20`, `yolo_40`: `True` if YOLO may use the photo in that experiment
>
> PatchCore will use the rows where `split` is `train` and `is_defect` is `False`.
>
> Below are the first 10 train defect photos of split 1, in label order (some columns are hidden to keep the table small), and then the excluded photos.

In [15]:
split_1 = pd.read_csv(os.path.join(SPLITS_DIR, "split_1.csv"), dtype={"label_order": "Int64"}).fillna({"same_gum_as": ""})
train_defects = split_1[(split_1["split"] == "train") & split_1["is_defect"]]

show = ["image", "defect_types", "type_group", "label_order", "yolo_5", "yolo_10", "yolo_20", "yolo_40"]
train_defects.sort_values("label_order")[show].head(10)

,image,defect_types,type_group,label_order,yolo_5,yolo_10,yolo_20,yolo_40
550,chewinggum/Data/Images/Anomaly/047.JPG,chunk of gum missing,chunk of gum missing,1,True,True,True,True
545,chewinggum/Data/Images/Anomaly/042.JPG,"chunk of gum missing,corner missing",corner missing,2,True,True,True,True
560,chewinggum/Data/Images/Anomaly/057.JPG,"chunk of gum missing,scratches,small cracks",scratches,3,True,True,True,True
579,chewinggum/Data/Images/Anomaly/076.JPG,similar colour spot,similar colour spot,4,True,True,True,True
556,chewinggum/Data/Images/Anomaly/053.JPG,"chunk of gum missing,small cracks",small cracks,5,True,True,True,True
553,chewinggum/Data/Images/Anomaly/050.JPG,chunk of gum missing,chunk of gum missing,6,False,True,True,True
533,chewinggum/Data/Images/Anomaly/030.JPG,"chunk of gum missing,corner missing",corner missing,7,False,True,True,True
570,chewinggum/Data/Images/Anomaly/067.JPG,scratches,scratches,8,False,True,True,True
600,chewinggum/Data/Images/Anomaly/097.JPG,similar colour spot,similar colour spot,9,False,True,True,True
563,chewinggum/Data/Images/Anomaly/060.JPG,"chunk of gum missing,corner missing,small cracks",small cracks,10,False,True,True,True


In [16]:
split_1.loc[split_1["split"] == "excluded", ["image", "split", "same_gum_as", "yolo_40"]]

,image,split,same_gum_as,yolo_40
113,chewinggum/Data/Images/Normal/113.JPG,excluded,chewinggum/Data/Images/Normal/112.JPG,False
216,chewinggum/Data/Images/Normal/216.JPG,excluded,chewinggum/Data/Images/Normal/215.JPG,False
265,chewinggum/Data/Images/Normal/265.JPG,excluded,chewinggum/Data/Images/Normal/264.JPG,False
300,chewinggum/Data/Images/Normal/300.JPG,excluded,chewinggum/Data/Images/Normal/299.JPG,False
351,chewinggum/Data/Images/Normal/351.JPG,excluded,chewinggum/Data/Images/Normal/350.JPG,False
501,chewinggum/Data/Images/Normal/501.JPG,excluded,chewinggum/Data/Images/Normal/500.JPG,False


## Summary

> - VisA's ready-made splits do not fit this project (see "VisA's Own Splits"), so we made our own.
> - 6 normal photos are left out, because they show the same gum as another photo (notebook 1.2). They have the part `excluded` in every split file.
> - 3 split files are saved in `splits/`, one per seed. Each has 347 normal + 40 defect train photos, 50 + 10 validation photos and 100 + 50 test photos (see "Make 3 Splits").
> - In the table above, photos 1 to 5 come from 5 different type groups, and `yolo_5` is `True` only for them.
>
> **Next:** `1.4_check_splits.ipynb` checks that the split files are correct and that no data leaks between the parts.